# Concurso Docente UBA - Extracción Automática desde cv_es.yaml

Este notebook genera texto base para copiar/pegar en el nuevo sistema de concursos docentes.

Uso recomendado:
1. Ejecutar la celda 2 (setup).
2. Ejecutar cada celda de sección (a, b, c, ...).
3. Copiar el resultado y ajustar redacción final si hace falta.

In [ ]:
import yaml
from pathlib import Path
from datetime import date

DATA_PATH = Path("../cv_db/cv_es.yaml")

with DATA_PATH.open("r", encoding="utf-8") as f:
    cv_data = yaml.safe_load(f)

print(f"Datos cargados desde: {DATA_PATH.resolve()}")

def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def list_to_text(value):
    out = []
    for item in as_list(value):
        if isinstance(item, dict):
            for k, v in item.items():
                out.append(f"{k}: {v}")
        else:
            out.append(str(item))
    return [x.strip() for x in out if str(x).strip()]

def get_path(*keys, default=None):
    node = cv_data
    for key in keys:
        if not isinstance(node, dict) or key not in node:
            return [] if default is None else default
        node = node[key]
    return node

def print_title(title):
    bar = "=" * len(title)
    print(bar)
    print(title)
    print(bar)

def format_date(d):
    return str(d or "").strip()

def is_ongoing(date_str):
    return "actual" in str(date_str).lower()

today = date.today().isoformat()
print(f"Fecha de ejecución: {today}")

## a. TITULOS UNIVERSITARIOS OBTENIDOS

In [ ]:
print_title("a) TÍTULOS UNIVERSITARIOS OBTENIDOS")

titulos = get_path("Educación", "Educación Universitaria")
for i, t in enumerate(as_list(titulos), 1):
    nombre = t.get("name", "")
    institucion = t.get("location", "")
    periodo = format_date(t.get("date", ""))
    detalles = "; ".join(list_to_text(t.get("description")))

    print(f"{i}. {nombre}")
    print(f"   Facultad/Universidad: {institucion}")
    print(f"   Período: {periodo}")
    if detalles:
        print(f"   Detalles: {detalles}")
    print()

## b. ANTECEDENTES DOCENTES E ÍNDOLE DE LAS TAREAS

In [ ]:
print_title("b) ANTECEDENTES DOCENTES E ÍNDOLE DE LAS TAREAS")

docencia = get_path("Experiencia", "Docencia y Formación")

def naturaleza_designacion(nombre):
    low = str(nombre).lower()
    if "invitado" in low:
        return "Invitado"
    if "facilitador" in low:
        return "Facilitador"
    if "co-organizador" in low or "organizador" in low:
        return "Organizador / Docente"
    return "Regular"

def contains_supervision(nombre, descripcion_list):
    """Check if item is about supervision (to optionally filter out)"""
    low_name = str(nombre).lower()
    supervision_keywords = ("supervisión", "supervision", "mentor")
    if any(k in low_name for k in supervision_keywords):
        return True
    for desc in as_list(descripcion_list):
        if isinstance(desc, dict):
            continue
        if any(k in str(desc).lower() for k in supervision_keywords):
            return True
    return False

# Sección 1: Docencia formal (excluyendo supervisión pura)
print("DOCENCIA Y ACTIVIDADES FORMATIVAS:")
for i, item in enumerate(as_list(docencia), 1):
    nombre = item.get("name", "")
    periodo = format_date(item.get("date", ""))
    institucion = item.get("location", "")
    designacion = naturaleza_designacion(nombre)
    tareas = list_to_text(item.get("description"))

    # Opcional: saltar supervisiones puras
    if contains_supervision(nombre, tareas):
        continue

    print(f"{i}. {nombre}")
    print(f"   Institución/lugar: {institucion}")
    print(f"   Período: {periodo}")
    print(f"   Naturaleza de la designación: {designacion}")
    print("   Tareas desarrolladas:")
    for t in tareas:
        print(f"   - {t}")
    print()

# Sección 2: Materiales publicados en Zenodo
print("\n" + "=" * 60)
print("MATERIALES DOCENTES Y RECURSOS PUBLICADOS")
print("=" * 60)

zenodo_items = []

# Buscar Zenodo en Docencia y Formación
for item in as_list(docencia):
    detalles = list_to_text(item.get("description"))
    for det in detalles:
        if "zenodo" in det.lower():
            zenodo_items.append({
                "source": "Docencia",
                "title": item.get("name", ""),
                "date": item.get("date", ""),
                "location": item.get("location", ""),
                "material": det
            })

# Buscar Zenodo en Posters y Presentaciones Orales
posters = get_path("Producción", "Posters y Presentaciones Orales")
for item in as_list(posters):
    detalles = list_to_text(item.get("description"))
    for det in detalles:
        if "zenodo" in det.lower():
            zenodo_items.append({
                "source": "Presentación",
                "title": item.get("title", ""),
                "event": item.get("event", ""),
                "date": item.get("date", ""),
                "material": det
            })

if zenodo_items:
    for i, zitem in enumerate(zenodo_items, 1):
        if "title" in zitem:
            print(f"\n{i}. {zitem['title']}")
            if "date" in zitem and zitem['date']:
                print(f"   Año: {zitem['date']}")
            if "location" in zitem and zitem['location']:
                print(f"   Institución: {zitem['location']}")
            if "event" in zitem and zitem['event']:
                print(f"   Evento: {zitem['event']}")
            print(f"   Material: {zitem['material']}")
else:
    print("No se encontraron materiales en Zenodo en la CV actual.")

print("\n" + "=" * 60)
print("(Puedes copiar cualquiera de estas dos secciones o ambas)")
print("=" * 60)


## c. ANTECEDENTES CIENTÍFICOS Y PUBLICACIONES

In [ ]:
print_title("c) ANTECEDENTES CIENTÍFICOS")

print("PUBLICACIONES:")
publicaciones = get_path("Producción", "Publicaciones")
for i, p in enumerate(as_list(publicaciones), 1):
    autores = p.get("authors", "")
    fecha = format_date(p.get("date", ""))
    titulo = p.get("title", "")
    revista = p.get("journal", "")
    detalles = "; ".join(list_to_text(p.get("description")))

    print(f"{i}. {autores} ({fecha}). {titulo}. {revista}.")
    if detalles:
        print(f"   Detalles: {detalles}")

print()
print("OTROS ANTECEDENTES RELACIONADOS CON LA ESPECIALIDAD (INVESTIGACIÓN):")
investigacion = get_path("Experiencia", "Investigación")
for i, item in enumerate(as_list(investigacion), 1):
    nombre = item.get("name", "")
    fecha = format_date(item.get("date", ""))
    lugar = item.get("location", "")
    desc = "; ".join(list_to_text(item.get("description")))
    print(f"{i}. {nombre}. {lugar}. Período: {fecha}.")
    if desc:
        print(f"   Detalles: {desc}")
    print()

## d. CURSOS, CONFERENCIAS Y TRABAJOS DE INVESTIGACIÓN

In [ ]:
print_title("d) CURSOS DE ESPECIALIZACIÓN, CONFERENCIAS Y TRABAJOS")

cursos = get_path("Cursos y Congresos")
for i, c in enumerate(as_list(cursos), 1):
    nombre = c.get("name", "")
    fecha = format_date(c.get("date", ""))
    lugar = c.get("location", "")
    duracion = c.get("extension", "")
    idioma = c.get("language", "")
    desc = "; ".join(list_to_text(c.get("description")))

    print(f"{i}. {nombre}.")
    print(f"   Lapso: {fecha}")
    print(f"   Lugar: {lugar}")
    if duracion:
        print(f"   Duración/carga horaria: {duracion}")
    if idioma:
        print(f"   Idioma: {idioma}")
    if desc:
        print(f"   Actividad: {desc}")
    print()

print("TRABAJOS DE INVESTIGACIÓN (EDITOS / PREPRINTS):")
for i, p in enumerate(as_list(get_path("Producción", "Publicaciones")), 1):
    journal = str(p.get("journal", "")).lower()
    if "biorxiv" in journal or "arxiv" in journal:
        print(f"- {p.get('title','')} ({p.get('date','')}) - {p.get('journal','')}")

## e. PARTICIPACIÓN EN CONGRESOS O ACONTECIMIENTOS SIMILARES

In [ ]:
print_title("e) PARTICIPACIÓN EN CONGRESOS O ACONTECIMIENTOS SIMILARES")

posters = get_path("Producción", "Posters y Presentaciones Orales")
for i, item in enumerate(as_list(posters), 1):
    titulo = item.get("title", "")
    evento = item.get("event", "")
    lugar = item.get("location", "")
    fecha = format_date(item.get("date", ""))
    autores = item.get("authors", "")
    calidad = "; ".join(list_to_text(item.get("description")))

    print(f"{i}. {evento}.")
    print(f"   Título/actividad: {titulo}")
    print(f"   Lugar: {lugar}")
    print(f"   Lapso: {fecha}")
    if autores:
        print(f"   Autores/representación: {autores}")
    if calidad:
        print(f"   Calidad de participación: {calidad}")
    print()

## f. ACTUACIÓN EN INSTITUCIONES Y CARGOS EN SECTOR PÚBLICO/PRIVADO

In [ ]:
print_title("f.1) ACTUACIÓN EN UNIVERSIDADES E INSTITUTOS")

# Palabras clave para filtrar: bioimage analyst, postdoc con beca, freelance
claves_bioimage = ("bioimage", "bioimágenes", "analista")
claves_beca = ("beca", "funded", "subsidiado", "vetenskapsrådet", "seal of excellence", "sello de excelencia")

print("ACTIVIDADES COMO ANALISTA DE BIOIMÁGENES EN ARGENTINA Y SUECIA:")
idx = 1

# Sección Académico: buscar "Analista de bioimágenes"
for item in as_list(get_path("Experiencia", "Académico")):
    nombre = item.get("name", "")
    if any(k in nombre.lower() for k in claves_bioimage):
        fecha = format_date(item.get("date", ""))
        lugar = item.get("location", "")
        desc = "; ".join(list_to_text(item.get("description")))
        print(f"{idx}. {nombre}")
        print(f"   Organismo/entidad: {lugar}")
        print(f"   Lapso: {fecha}")
        if desc:
            print(f"   Tareas: {desc}")
        print()
        idx += 1

# Sección Investigación: buscar postdocs con financiamiento en Sweden/instituciones relevantes
print("POSTDOCTORADOS Y BECAS DE INVESTIGACIÓN:")
for item in as_list(get_path("Experiencia", "Investigación")):
    nombre = item.get("name", "")
    lugar = item.get("location", "")
    desc_list = list_to_text(item.get("description"))
    
    # Detectar si tiene financiamiento mencionado o si está en Sweden/institución relevante
    tiene_beca = any(k in " ".join(desc_list).lower() for k in claves_beca)
    en_suecia = "suecia" in lugar.lower() or "sweden" in lugar.lower()
    
    if tiene_beca or en_suecia:
        fecha = format_date(item.get("date", ""))
        print(f"{idx}. {nombre}")
        print(f"   Institución/lugar: {lugar}")
        print(f"   Lapso: {fecha}")
        if desc_list:
            for d in desc_list:
                print(f"   - {d}")
        print()
        idx += 1

# Sección Profesionales: buscar trabajo freelance en análisis de bioimágenes
print("=" * 60)
print("ACTIVIDADES PROFESIONALES COMO ANALISTA (Consultoría/Freelance):")
for i, item in enumerate(as_list(get_path("Experiencia", "Profesionales")), 1):
    nombre = item.get("name", "")
    fecha = format_date(item.get("date", ""))
    lugar = item.get("location", "")
    desc = "; ".join(list_to_text(item.get("description")))
    print(f"{i}. {nombre}")
    print(f"   Entidad/lugar: {lugar}")
    print(f"   Lapso: {fecha}")
    if desc:
        print(f"   Funciones: {desc}")
    print()


## g. FORMACIÓN DE RECURSOS HUMANOS

In [ ]:
print_title("g) FORMACIÓN DE RECURSOS HUMANOS")

docencia = as_list(get_path("Experiencia", "Docencia y Formación"))
claves_supervision = ("supervisión", "supervision", "mentor", "estudiante")
claves_becas = ("beca", "funded", "subsidiado", "seal of excellence", "sello de excelencia")

# Sección 1: Supervisión de estudiantes
print("SUPERVISIÓN Y FORMACIÓN DE ESTUDIANTES:")
idx_supervision = 1
for item in docencia:
    nombre = str(item.get("name", "")).lower()
    descripcion = list_to_text(item.get("description"))
    texto_desc = " ".join(descripcion).lower()
    
    # Detectar si es un item de supervisión
    is_supervision = any(c in nombre for c in claves_supervision) or any(c in texto_desc for c in claves_supervision)
    
    if is_supervision:
        print(f"{idx_supervision}. {item.get('name','')}")
        print(f"   Institución/lugar: {item.get('location','')}")
        print(f"   Lapso: {item.get('date','')}")
        if descripcion:
            for d in descripcion:
                print(f"   - {d}")
        print()
        idx_supervision += 1

if idx_supervision == 1:
    print("(No se encontraron supervisiones de estudiantes)")
    print()



## h. SÍNTESIS DE APORTES ORIGINALES

Durante el período 2016-actualidad he realizado aportes originales en la interfaz entre biofísica, microscopía avanzada y análisis cuantitativo de bioimágenes, con impacto tanto metodológico como aplicado. Estos aportes se desarrollaron en instituciones de Argentina y Suecia, en contextos de investigación interdisciplinaria y colaboración internacional.

En mi etapa doctoral (2016-2021, CONICET - Universidad de Buenos Aires) trabajé en la cuantificación de procesos dinámicos en sistemas biológicos complejos, integrando modelado físico y análisis computacional para describir fenómenos de organización celular y tisular. Esta línea consolidó una base conceptual y técnica en adquisición, procesamiento y análisis de señales de imagen.

Durante mi etapa posdoctoral (2022-2024, Karolinska Institutet; financiamiento Vetenskapsrådet y reconocimiento Marie Skłodowska-Curie Actions Seal of Excellence) profundicé en el desarrollo y la validación de metodologías reproducibles para el análisis de bioimágenes, incorporando herramientas de ciencia de datos e inteligencia artificial para extraer información biológica cuantitativa de alta resolución.

En el plano de transferencia y desarrollo tecnológico (2024-2025, BioImage Informatics Facility - SciLifeLab/NBIS, Uppsala Universitet), diseñé y optimicé pipelines de análisis para múltiples modalidades de microscopía, contribuyendo a la estandarización de flujos reproducibles y al soporte metodológico de proyectos de investigación en distintas áreas biológicas.

Como resultado, mis aportes originales se expresan en:

- Integración de fundamentos físicos con métodos computacionales e IA para análisis de imágenes.
- Desarrollo de estrategias reproducibles y transferibles para investigación de frontera.
- Aplicación de estas metodologías en colaboraciones internacionales y publicaciones científicas de alto impacto.
- Formación de capacidades en la comunidad mediante docencia especializada, talleres y recursos abiertos.

## i. SÍNTESIS DE ACTUACIÓN PROFESIONAL Y/O EXTENSIÓN UNIVERSITARIA

Mi actuación profesional reciente se centró en el desarrollo y optimización de metodologías avanzadas de análisis de imágenes para instituciones académicas de investigación en Argentina y Suecia. Entre 2024 y 2025, me desempeñé como analista de bioimágenes en BioImage Informatics Facility (Science for Life Laboratory, Uppsala Universitet y National Bioinformatics Infrastructure Sweden), donde diseñé flujos de análisis automatizados para una amplia gama de problemas biológicos y modalidades de imagen.

Este trabajo no se limita a procesar imágenes: parte de la pregunta biológica y la conecta con la teoría fotofísica de la señal para cuantificar variables biológicas relevantes (por ejemplo, concentración relativa, localización subcelular, dinámica espacio-temporal y relaciones entre marcadores). Sobre esa base cuantitativa, también contribuye directamente a mejorar el diseño experimental, ajustando condiciones de adquisición, controles, resolución temporal y espacial, y estrategia de muestreo para aumentar la calidad inferencial de los resultados.

Además, el análisis se integra con el modelado de la dinámica del sistema en estudio, permitiendo formular y contrastar hipótesis mecanísticas a continuación del experimento. En este enfoque iterativo, los hallazgos computacionales retroalimentan el diseño experimental y el diseño experimental mejora la capacidad del modelo para describir y predecir el comportamiento biológico.

Paralelamente, he participado activamente en actividades de extensión universitaria y transferencia de conocimiento, destacándose:

- Dictado de cursos y talleres intensivos: Coordiné e impartí cursos sobre análisis de bioimágenes, microscopía avanzada e inteligencia artificial aplicada a la biología en instituciones como Karolinska Institutet, Universidad de Gotemburgo, Center for Cellular Imaging y universidades de América Latina (KhipuX Quito 2026).
- Divulgación científica: Participación como expositor en eventos públicos y jornadas de ciencia abierta, promoviendo la alfabetización digital y el análisis cuantitativo en bioimágenes.
- Generación de recursos educativos abiertos: Desarrollo y publicación de materiales docentes (tutoriales, conjuntos de datos, código fuente) en plataformas como Zenodo, facilitando la adopción de metodologías reproducibles por la comunidad científica internacional.
- Participación en redes científicas: Formo parte de GloBIAS (Global Bioimage Analysts) prácticamente desde sus inicios, coordinando actividades de introducción de la sociedad ante la comunidad científica, el relevamiento del estado del campo a nivel mundial y la organización de actividades educativas. En paralelo, trabajo desde Latin America BioImaging (LABI), articulando sus acciones con GloBIAS para fortalecer las capacidades regionales en bioimágenes en América Latina.

Esta actuación refleja mi compromiso con la democratización del conocimiento, la ciencia abierta y la construcción de capacidades científicas en contextos internacionales e interdisciplinarios.

## j. OTROS ELEMENTOS DE JUICIO VALIOSOS

**Becas y financiamientos internacionales acreditados:**
- **Postdoctorado financiado por Vetenskapsrådet** (Swedish Research Council): Apoyo a investigación postdoctoral en Karolinska Institutet (2022–2024).
- **Beca "Seal of Excellence" de Marie Skłodowska-Curie Actions** (2022–2024): Reconocimiento de excelencia científica para investigación en desarrollo embrionario.
- **Beca doctoral CONICET** (2016–2021): Soporte para investigación doctoral en ciencias físicas.

**Participación y colaboraciones internacionales:**
- Miembro activo de **GloBIAS (Global Bioimage Analysts)**: Red internacional dedicada al fortalecimiento de capacidades en análisis de bioimágenes, con participación en workshops y acciones de capacitación en América Latina.
- Colaboraciones con investigadores de instituciones de prestigio en Europa (Karolinska Institutet, Max Planck Institute, Institut Curie) y América Latina.
- Experiencia de investigación en Argentina, Suecia y Alemania.

**Publicaciones de alto impacto:**
- Coautor en artículos publicados en revistas de alcance internacional como *Science*, *Nature Methods*, *Development*, *Journal of Cell Science*.
- Contribuciones a proyectos colaborativos multinacionales en bioinformática e imagen (AI4Life Open Calls, GloBIAS initiatives).

**Reconocimiento de pares:**
- Convocado para preparar y dictar diversos cursos y talleres en universidades e institutos internacionales de Ecuador, Chile, Uruguay, Suecia y Argentina.
- Convocado como jurado de dos tesis de licenciatura en la Universidad de Buenos Aires.

## k. PLAN DE LABOR DOCENTE, INVESTIGACIÓN Y EXTENSIÓN

##### **Sus puntos de vista sobre temas básicos de su campo del conocimiento que deben transmitirse a los alumnos; la importancia relativa y ubicación de su área en el currículo de la carrera. Medios que propone para mantener actualizada la enseñanza y para llevar a la práctica los cambios que sugiere.**

En el marco de mi postulación como **Profesor Adjunto** en la carrera de Física, mi plan docente se centrará en el beneficio que trae la sinergia entre plantear un modelo de la dinámica del sistema bajo estudio, comprendiendo el rol que cumplen los parámetros del sistema, y combinarlo con análisis de datos y ciencia de datos que permita una evaluación estadística robusta, para luego poder mejorar el diseño experimental y la adquisición de datos. Este enfoque apunta a comprobar de manera cuantitativa si el modelo planteado describe adecuadamente el sistema bajo estudio y, cuando corresponda, a refinar el modelo y/o el diseño experimental sobre la base de evidencia.

Cuento con experiencia en **física interdisciplinaria**, en particular en **biofísica**, y esa trayectoria orienta mi propuesta de enseñanza. En el área de biofísica donde me desarrollo, mis conocimientos como físico me permiten traducir datos de detectores y sensores de cámaras en variables fotofísicas, que luego puedo transformar en variables de interés biológico a partir de su comprensión física. Esta perspectiva será parte central de los contenidos, de las prácticas y de la actualización permanente de la enseñanza.

Para llevar estos cambios a la práctica, propondré actividades en Laboratorios de Enseñanza Superior que integren diseño experimental, adquisición de datos robusta, análisis reproducible y documentación completa de parámetros y decisiones metodológicas. En este contexto, el auge de los modelos fundacionales de visión y de los agentes de inteligencia artificial ha revolucionado mi área específica; por eso no solo me mantengo al día con estos avances, sino que también imparto conocimientos sobre estas herramientas y su uso crítico en problemas reales de biofísica y análisis de bioimágenes. Asimismo, discuto y enseño la importancia de la reproducibilidad y cómo utilizar herramientas de versionado de datos y de análisis para garantizarla, a través de herramientas como Git.

## l. PLAN DE INVESTIGACIÓN CIENTÍFICA Y TECNOLÓGICA Y/O EXTENSIÓN

Mi plan de investigación se centra en la **intersección entre la física, la biofísica, la inteligencia artificial y el análisis cuantitativo de imágenes**, con el objetivo de desarrollar metodologías innovadoras que combinen conocimientos de principios físicos subyacentes con herramientas computacionales avanzadas.

### **Pilares fundamentales:**

**1. Integración de la física en el análisis de imágenes con IA**

A diferencia de enfoques puramente guiados por datos, propongo incorporar el conocimiento del origen físico de las señales (fotofísica de fluorescencia, dispersión óptica) para optimizar algoritmos de aprendizaje automático. Esto permitirá extraer información biofísica precisa (concentraciones de proteínas, dinámicas moleculares, parámetros de colocalización) que trascienda la mera segmentación de estructuras.

**2. Validación experimental y contraste de modelos teóricos**

Las cuantificaciones obtenidas mediante análisis avanzado de imágenes servirán para validar y refinar modelos matemáticos que describen procesos biológicos complejos. Este circuito de retroalimentación entre observación y teoría permitirá transitar desde descripciones cualitativas a comprensiones predictivas y cuantitativas de fenómenos en desarrollo embrionario y organogénesis.

**3. Optimización del flujo experimental mediante análisis de datos**

Trabajando con servicios de adquisición de imágenes (Centro de Microscopías Avanzadas, NBIS), estableceré un circuito de retroalimentación donde los hallazgos del análisis inspiren mejoras en el diseño experimental y en los parámetros de adquisición. Esto acelerará y mejorará significativamente la calidad de la investigación.

**4. Análisis multi-escala e integración de datos heterogéneos**

Aprovecharé experiencia previa en ómicas y análisis de grandes volúmenes de datos para enriquecer el análisis de microscopía. Combinaré cuantificaciones de imágenes (morfología, intensidad de reporteros) con datos genómicos, transcriptómicos y bioquímicos para construir visiones holísticas e integradas de los sistemas biológicos estudiados.

### **Líneas de investigación prioritarias:**

- Desarrollo de métodos de análisis de imágenes guiados por física para microscopía de fluorescencia.
- Aplicación de redes neuronales especializadas para la cuantificación de fenotipos en organoides y modelos 3D.
- Análisis cuantitativo de procesos del desarrollo embrionario mediante integración de imaging y transcriptómica.
- Transferencia de metodologías reproducibles a laboratorios de investigación clínica y académica.

### **Impacto esperado:**

Esta investigación contribuirá tanto al avance del conocimiento fundamental en biofísica y biología del desarrollo como al desarrollo de herramientas tecnológicas transferibles que fortalezcan las capacidades nacionales e internacionales en análisis cuantitativo de imágenes.